# 04 - Engenharia de Features (Lags, Calendário e Anti-Leakage)

**Objetivo:** Transformar a Base Mestra em um conjunto de preditores de alta dimensão, integrando contextuais (eventos/reuniões), cardápio e dependências temporais.

**Rigor Técnico:** 
- **Anti-Leakage:** Todas as variáveis baseadas no alvo (`total_servido`) ou clima utilizam obrigatoriamente um deslocamento temporal (`shift(1)`).
- **Consistência:** Unificação das etapas acadêmicas (0 e 1 unificadas para 1 - 1º Bimestre).

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# Configurações
BASE_MESTRA = '../data/base_mestra_consolidada.csv'
SAIDA_FEATURES = '../data/base_features_final.csv'

# 2. Carregamento e Ordenação
df = pd.read_csv(BASE_MESTRA)
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)

print(f"Base carregada: {len(df)} registros.")

Base carregada: 198 registros.


## 1. Tratamento de Sazonalidade e Etapas
Unificação para consistência com a EDA e criação de dummies.

In [2]:
# Unificação de Etapas (Conforme definido na EDA)
if 'etapa' in df.columns:
    df['etapa'] = df['etapa'].replace(0, 1)
    df = pd.get_dummies(df, columns=['etapa'], prefix='etapa_acad', drop_first=False)

# Features Cíclicas de Calendário (Seno/Cosseno)
df['dia_semana'] = df['data'].dt.dayofweek
df['dia_semana_sin'] = np.sin(2 * np.pi * df['dia_semana'] / 7)
df['dia_semana_cos'] = np.cos(2 * np.pi * df['dia_semana'] / 7)
df['eh_segunda'] = (df['dia_semana'] == 0).astype(int)
df['eh_sexta'] = (df['dia_semana'] == 4).astype(int)

# Features Binárias de Impacto
colunas_binarias = ['eh_evento_especial', 'eh_reuniao_impacto', 'vespera_feriado', 'eh_feriado']
for col in colunas_binarias:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)
    else:
        df[col] = 0

for col in df.columns:
    if col.startswith('etapa_acad'):
        df[col] = df[col].astype(int)

## 2. Encoding do Cardápio
Transformando a Proteína Principal em variáveis binárias (One-Hot Encoding).

In [3]:
# Encoding de Proteína Principal
print(f"Proteínas para Encoding: {df['proteina_principal'].unique()}")
df = pd.get_dummies(df, columns=['proteina_principal'], prefix='prot', drop_first=False)

# Converter booleanos para int
for col in df.columns:
    if col.startswith('prot_'):
        df[col] = df[col].astype(int)

Proteínas para Encoding: ['BOVINA' 'AVE' 'PEIXE' 'SUINA' 'FEIJOADA']


## 3. Dependências Temporais (Lags e Médias Móveis)
**CRÍTICO:** Uso de `.shift(1)` para evitar vazamento do alvo do dia atual.

In [4]:
# 3.1 Lags de Reservas
for lag in [1, 2, 3, 7]:
    df[f'total_reservas_lag_{lag}'] = df['total_reservas'].shift(lag)

# 3.2 Lags de Erro de Planejamento (O erro de ONTEM ajuda a prever HOJE)
df['erro_planejamento_ontem'] = (df['total_servido'] - df['total_reservas']).shift(1)

# 3.3 Médias Móveis (Baseadas apenas no passado)
df['reservas_media_movel_3d'] = df['total_reservas'].shift(1).rolling(window=3).mean()
df['reservas_media_movel_5d'] = df['total_reservas'].shift(1).rolling(window=5).mean()

# 3.4 Variáveis Climáticas Lagged
df['chuva_ontem'] = df['chuva_soma'].shift(1)
df['temp_media_ontem'] = df['temp_media'].shift(1)

## 4. Features de Interação
Capturando o efeito combinado de reservas com eventos e reuniões.

In [5]:
df['interacao_reserva_evento'] = df['total_reservas'] * df['eh_evento_especial']
df['interacao_reserva_reuniao'] = df['total_reservas'] * df['eh_reuniao_impacto']
df['interacao_reserva_v_feriado'] = df['total_reservas'] * df['vespera_feriado']

## 5. Limpeza e Exportação da Base de Features

In [6]:
# Remover linhas iniciais com NaNs (causados pelos lags)
df_features = df.dropna().copy()

# Remover colunas que não são preditoras ou redundantes
cols_remover = ['evento', 'chuva_soma', 'temp_media', 'temp_max', 'temp_min', 'vento_max', 
                'preparo_principal', 'reservou_e_comeu', 'nao_reservou_e_comeu', 'reservou_e_nao_comeu']
df_features = df_features.drop(columns=[c for c in cols_remover if c in df_features.columns])

# Salvar
df_features.to_csv(SAIDA_FEATURES, index=False)

print(f"✨ Engenharia de Features concluída com {len(df_features.columns)} colunas.")
print(f"📈 Base Final salva em: {SAIDA_FEATURES}")
df_features.head()

✨ Engenharia de Features concluída com 35 colunas.
📈 Base Final salva em: ../data/base_features_final.csv


,data,eh_feriado,vespera_feriado,eh_reuniao_impacto,tem_refeicao,eh_evento_especial,proteina_vegetariana,total_reservas,total_servido,etapa_acad_1,...,total_reservas_lag_3,total_reservas_lag_7,erro_planejamento_ontem,reservas_media_movel_3d,reservas_media_movel_5d,chuva_ontem,temp_media_ontem,interacao_reserva_evento,interacao_reserva_reuniao,interacao_reserva_v_feriado
7,2023-08-01,0,0,0,1,0,VEGETAL,131.0,118.0,1,...,132.0,121.0,-29.0,133.333333,132.6,0.0,21.99,0.0,0.0,0.0
8,2023-08-02,0,0,0,1,0,VEGETAL,84.0,46.0,1,...,115.0,174.0,-13.0,133.000000,132.6,0.0,17.98,0.0,0.0,0.0
9,2023-08-03,0,0,0,1,0,OVO,114.0,109.0,1,...,153.0,131.0,-38.0,122.666667,123.0,0.0,18.67,0.0,0.0,0.0
10,2023-08-04,0,0,0,1,0,VEGETAL,84.0,112.0,1,...,131.0,132.0,-5.0,109.666667,119.4,0.0,18.42,0.0,0.0,0.0
11,2023-08-07,0,0,0,1,0,VEGETAL,82.0,103.0,1,...,84.0,132.0,28.0,94.000000,113.2,0.0,17.68,0.0,0.0,0.0
